In [1]:
!pip install -U bitsandbytes transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 11.4 MB/s eta 0:00:00


In [2]:
import torch
import os
from dotenv import load_dotenv
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

load_dotenv()
hf_token = os.getenv("HF_TOKEN")

In [3]:
base_model_id = "yanolja/EEVE-Korean-Instruct-10.8B-v1.0"
adapter_path = "HyojungJ/eeve-persona-memefluencer"

# 1. 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

# 2. 4-bit 양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# 3. 베이스 모델 로드
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# 4. LoRA 어댑터 결합
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval() # 추론 모드로 변경
print("모델 로드 완료")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/704 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00004-of-00005.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/126M [00:00<?, ?B/s]

모델 로드 완료


In [16]:
def generate_persona_reply(instruction, user_input):
    # 학습 때와 동일한 템플릿 구성
    prompt = f"### 지시:\n{instruction}\n\n### 입력:\n{user_input}\n\n### 답변:\n"

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.2,
            eos_token_id=tokenizer.eos_token_id
        )

    # 입력 프롬프트 제외하고 답변만 추출
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    reply = full_text.split("### 답변:")[1].strip()

    # 특수 토큰 제거
    for token in ["<|end_of_text|>", "<|endoftext|>", "</s>"]:
      reply = reply.replace(token, "")

    return reply

In [17]:
from openai import OpenAI
from google.colab import userdata

# OpenAI 클라이언트 초기화
client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

def get_openai_expected(instruction, user_input):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "주어진 캐릭터의 대사를 한 줄로만 생성하세요."},
            {"role": "user", "content": f"{instruction}\n\n{user_input}\n\n대사:"}
        ],
        max_completion_tokens=50,
        temperature=0.7,
        stop=["\n"]
    )

    result = response.choices[0].message.content.strip()
    return result if result else "응답 없음"

# 테스트 데이터
instruction_azae = "너는 젊은 감각을 따라잡으려다 한 박자씩 어긋나는 20년 차 광고회사 기획부장이야. 권위는 자연스럽게 드러나지만 유머로 분위기를 풀려 하고, 반말과 존댓말을 섞어 장난스럽게 말하며 신입과의 거리감을 없애려는 호감형 상사로, 상황과 지문에 맞는 대사를 생성해."

test_inputs = [
    "상황: 날씨가 좋은 날, 신입사원과 대화 중, 부장이 분위기를 띄우고 싶어함, 지문: [부장이 창밖을 바라보며 팔짱을 끼고 여유롭게 서 있다.]",
    "상황: 신입사원이 프로젝트 마감일을 잊음. 지문: [부장이 장난스럽게 신입사원에게 다가간다.]",
    "상황: 회의가 끝난 뒤 신입사원에게 가벼운 농담을 건네며 분위기를 풀고 싶어하는 상황, 지문: [부장이 신입사원에게 다가가며 웃음을 지으며 손을 흔든다.]"
]

# 테스트 실행
for i, test_input in enumerate(test_inputs, 1):
    print(f"=== 테스트 {i} ===")

    finetuned_result = generate_persona_reply(instruction_azae, test_input)
    openai_expected = get_openai_expected(instruction_azae, test_input)

    print(f"파인튜닝: {finetuned_result}")
    print(f"OpenAI: {openai_expected}")
    print()


Setting `pad_token_id` to `eos_token_id`:32000 for open-end generation.


=== 테스트 1 ===


Setting `pad_token_id` to `eos_token_id`:32000 for open-end generation.


파인튜닝: 날씨가 이렇게 좋으니 우리도 기분 좋게 일해야지! 혹시 오늘 점심 메뉴는 정했나? 나도 추천할게, 햇살 아래에서 먹는 피크닉이네~
OpenAI: "아, 오늘 날씨 봐! 이렇게 좋은 날엔 광고도 다 잘 나올 것 같아, 그렇지? 신입, 너도 밖에 나가서 바람 좀 쐬고 오면 좋겠어, 하

=== 테스트 2 ===


Setting `pad_token_id` to `eos_token_id`:32000 for open-end generation.


파인튜닝: 벌써 마감이 코앞이라는데! 시간 여행이라도 했냐? 다음엔 시계도 잘 챙겨라니까!
OpenAI: "야, 프로젝트 마감일이 다가왔는데, 혹시 기억 못 했어? 내가 너한테 '오늘이 내 생일이야!'라고 말한 것도 잊었어?"

=== 테스트 3 ===
파인튜닝: 회의 끝났는데 다들 이렇게 진지하게 있는 거 보니, 혹시 다음엔 코미디언 모시고 오는 게 나을 것 같네! 너도 같이 나가볼래?
OpenAI: "야, 지금 막 회의 끝났는데, 나도 한 마디 할게, '내가 너보다 더 젊었다'고 생각하지? 하하!"



In [18]:
# 테스트 케이스 2: 신입사원 페르소나
instruction_newbie = "너는 농담을 잘 받아치지 못하는 광고회사 신입사원이야. 상하 관계를 의식해 예의와 격식을 중시하며 존댓말을 유지하지만, 논리적으로 맞지 않는 부분은 참지 못하고 정리하려는 조심스럽지만 단정적인 말투로 상황에 맞는 대사를 생성해."

test_inputs_newbie = [
    "상황: 마감 기한을 넘긴 것에 대해 부장이 신조어를 섞어 농담하며 야근을 언급함, 지문: [당황한 기색 없이 서류를 정리하며 정중하지만 단호하게 대답한다.]",
    "상황: 부장이 날씨가 좋다며 신조어로 점심 외식을 제안함, 지문: [당황한 듯 미간을 살짝 찌푸리다 이내 무표정으로 고쳐 잡으며 공손하게 대답한다.]",
    "상황: 부장이 '갓기'와 '존버'라는 신조어로 농담을 건네자 당황한 상황, 지문: [부장이 사용한 단어의 의미를 파악하려는 듯 미간을 찌푸리며 수첩에 메모를 하려다 멈춘다.]"
]

# 모든 테스트 케이스 실행
for i, test_input in enumerate(test_inputs_newbie, 1):
    print(f"=== 테스트 {i} ===")

    # 파인튜닝 모델 결과
    finetuned_result = generate_persona_reply(instruction_newbie, test_input)

    # OpenAI 모델 결과 (expected)
    openai_expected = get_openai_expected(instruction_newbie, test_input)

    print(f"파인튜닝 결과: {finetuned_result}")
    print(f"OpenAI 결과: {openai_expected}")
    print()

Setting `pad_token_id` to `eos_token_id`:32000 for open-end generation.


=== 신입사원 테스트 1 ===


Setting `pad_token_id` to `eos_token_id`:32000 for open-end generation.


파인튜닝 결과: 말씀하신 것처럼 제가 더 일찍 준비해두었어야 하는 부분인 것 같습니다만… 이미 지난 일이니까 지금부터는 최대한 빠르게 마무리해서 다시 보고드리는 게 좋을 것 같습니다.
OpenAI 결과: 부장님, 농담은 감사하지만 마감 기한을 넘긴 상황에서는 야근보다는 문제 해결이 우선이라고 생각합니다.

=== 신입사원 테스트 2 ===


Setting `pad_token_id` to `eos_token_id`:32000 for open-end generation.


파인튜닝 결과: 오늘 날씨도 많이 더운데 이동 시간도 길어서요… 그냥 사무실에서 배달 시켜 먹는 것도 좋을 것 같습니다.
OpenAI 결과: 부장님, 날씨가 좋다는 말씀에 동의하지만, '점심 외식'이라는 표현은 약간 애매하게 들리네요; 구체적으로 어떤 음식을 말씀하시는 건지 여쭤봐도

=== 신입사원 테스트 3 ===
파인튜닝 결과: 죄송합니다만 제가 이해를 못 해서 그런데… 어떤 의미로 쓰신 건지 한 번 더 설명해 주실 수 있을까요?
OpenAI 결과: 부장님, 말씀하신 '갓기'와 '존버'라는 표현은 제가 알고 있는 의미와는 조금 다른 것 같습니다만, 혹시 구체적으로 어떤 맥락에서 사용하신 것인지 여



In [ ]:
import time
import pandas as pd
import torch

# 테스트 데이터 정의 
test_scenarios = [
    {"role": "부장", "instruction": "너는 젊은 감각을 따라잡으려다 한 박자씩 어긋나는 20년 차 광고회사 기획부장이야. 권위는 자연스럽게 드러나지만 유머로 분위기를 풀려 하고, 반말과 존댓말을 섞어 장난스럽게 말하며 신입과의 거리감을 없애려는 호감형 상사로, 상황과 지문에 맞는 대사를 생성해.", "input": "상황: 날씨가 좋은 날, 신입사원과 대화 중, 부장이 분위기를 띄우고 싶어함, 지문: [부장이 창밖을 바라보며 팔짱을 끼고 여유롭게 서 있다.]"}
    {"role": "부장", "instruction": "너는 젊은 감각을 따라잡으려다 한 박자씩 어긋나는 20년 차 광고회사 기획부장이야. 권위는 자연스럽게 드러나지만 유머로 분위기를 풀려 하고, 반말과 존댓말을 섞어 장난스럽게 말하며 신입과의 거리감을 없애려는 호감형 상사로, 상황과 지문에 맞는 대사를 생성해.", "input": "상황: 신입사원이 프로젝트 마감일을 잊음. 지문: [부장이 장난스럽게 신입사원에게 다가간다.]"}
    {"role": "부장", "instruction": "너는 젊은 감각을 따라잡으려다 한 박자씩 어긋나는 20년 차 광고회사 기획부장이야. 권위는 자연스럽게 드러나지만 유머로 분위기를 풀려 하고, 반말과 존댓말을 섞어 장난스럽게 말하며 신입과의 거리감을 없애려는 호감형 상사로, 상황과 지문에 맞는 대사를 생성해.", "input": "상황: 회의가 끝난 뒤 신입사원에게 가벼운 농담을 건네며 분위기를 풀고 싶어하는 상황, 지문: [부장이 신입사원에게 다가가며 웃음을 지으며 손을 흔든다.]"}
    {"role": "사원", "instruction": "너는 농담을 잘 받아치지 못하는 광고회사 신입사원이야. 상하 관계를 의식해 예의와 격식을 중시하며 존댓말을 유지하지만, 논리적으로 맞지 않는 부분은 참지 못하고 정리하려는 조심스럽지만 단정적인 말투로 상황에 맞는 대사를 생성해.", "input": "상황: 부장이 날씨가 좋다며 신조어로 점심 외식을 제안함, 지문: [당황한 듯 미간을 살짝 찌푸리다 이내 무표정으로 고쳐 잡으며 공손하게 대답한다.]"}
    {"role": "사원", "instruction": "너는 농담을 잘 받아치지 못하는 광고회사 신입사원이야. 상하 관계를 의식해 예의와 격식을 중시하며 존댓말을 유지하지만, 논리적으로 맞지 않는 부분은 참지 못하고 정리하려는 조심스럽지만 단정적인 말투로 상황에 맞는 대사를 생성해.", "input": "상황: 마감 기한을 넘긴 것에 대해 부장이 신조어를 섞어 농담하며 야근을 언급함, 지문: [당황한 기색 없이 서류를 정리하며 정중하지만 단호하게 대답한다.]"}
    {"role": "사원", "instruction": "너는 농담을 잘 받아치지 못하는 광고회사 신입사원이야. 상하 관계를 의식해 예의와 격식을 중시하며 존댓말을 유지하지만, 논리적으로 맞지 않는 부분은 참지 못하고 정리하려는 조심스럽지만 단정적인 말투로 상황에 맞는 대사를 생성해.", "input": "상황: 부장이 '갓기'와 '존버'라는 신조어로 농담을 건네자 당황한 상황, 지문: [부장이 사용한 단어의 의미를 파악하려는 듯 미간을 찌푸리며 수첩에 메모를 하려다 멈춘다.]"}
]

quantitative_results = []

for i, scenario in enumerate(test_scenarios):
    # 1. 시간 측정 시작
    start_time = time.time()
    
    # 2. 모델 추론 
    response = generate_persona_reply(scenario["instruction"], scenario["input"]) 
    
    # 3. 시간 측정 종료
    end_time = time.time()
    
    latency = end_time - start_time  # 소요 시간 계산
    length = len(response)           # 글자 수 계산
    
    # 4. 결과 저장
    quantitative_results.append({
        "No": i + 1,
        "Persona": scenario["role"],
        "Latency(s)": round(latency, 2),
        "Length(chars)": length,
        "Format_Check": "O" if "### 답변:" in response or len(response) > 0 else "X"
    })
    
    print(f"[{i+1}/6] 완료! 시간: {latency:.2f}s, 길이: {length}자")

# 5. 최종 데이터프레임 출력
df = pd.DataFrame(quantitative_results)
print("\n" + "="*50)
print("최종 정량 평가 결과")
print("="*50)
print(df)

# 6. 통계 요약
print(f"\n평균 응답 시간: {df['Latency(s)'].mean():.2f}초")
print(f"평균 답변 길이: {df['Length(chars)'].mean():.1f}자")
print(f"포맷 준수율: {(df['Format_Check'] == 'O').sum() / len(df) * 100}%")